# Event displays — 2D, 3D, and animation

LUCiD's event-display gallery, all from `lucid.visualization` + the geometry's built-in 3D viewer
— **no inline display code** (the old notebooks each re-defined their own; those helpers now live
in the library). One simulated muon event, shown three ways.

In [ ]:
%matplotlib inline
import sys; sys.path.append('../..')
import jax, numpy as np
from lucid.geometry import generate_detector
from lucid.simulation import setup_event_simulator
from lucid.detector_params import ParticleParams
from lucid.visualization import create_detector_display, animate_event

GEOM, PHYS = '../../config/SK_like_geom_config.json', '../../config/SK_like_physics_config.json'
det = generate_detector(GEOM)
sim = setup_event_simulator(GEOM, 250_000, K=6, hit_mode='realistic', physics_config=PHYS,
                            default_detector_params=True, particle='muon',
                            n_cap=120, n_angular=200, n_height=120)
track = ParticleParams.from_cartesian(energy=1000., position=[0., 0., 0.], direction=[1., 0., 0.], t0=0.)
charge, time = (np.asarray(x) for x in jax.lax.stop_gradient(sim(track, jax.random.PRNGKey(0))))
print(f'{(charge>0).sum()} PMTs lit, total charge {charge.sum():.0f} pe')

### 1. 2D unrolled cylinder — barrel strip + top/bottom cap discs (per-PMT charge)

In [ ]:
display = create_detector_display(GEOM, sparse=False)
display(charge, time, file_name=None, perc_min=0.0, perc_max=99.5)

### 2. Time-evolution animation — hits arriving over time (writes a GIF)

In [ ]:
from IPython.display import Image
animate_event(det, charge, time, out_path='event.gif', n_frames=30, fps=12)
Image('event.gif')

### 3. Interactive 3D disc view (plotly) — every PMT as a disc on the tank surface

In [ ]:
lit = charge > 0
det.visualize_event_data_plotly_discs(np.where(lit)[0], charge[lit], time[lit],
                                      title='muon event — per-PMT charge')

All three read the same per-PMT `(charge, time)`. Swap `track` for any `ParticleParams`, or
feed real data (`lucid.sources.event_io.read_photon_data_from_photonsim`). The 2D display +
`animate_event` + `unroll_layout` all live in `lucid.visualization`; the 3D disc view is
`detector.visualize_event_data_plotly_discs`.